## 基本操作

In [97]:
import re
import os
import gc
import dask
import pymysql
import warnings
import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client
from sqlalchemy import create_engine
from datetime import datetime, timedelta
from pandas.errors import SettingWithCopyWarning
warnings.filterwarnings("ignore", category = UserWarning, module='openpyxl')
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message="pandas only supports SQLAlchemy connectable")
warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
warnings.filterwarnings('ignore', category=SettingWithCopyWarning)

In [98]:
from datetime import datetime
import pymysql

# 固定 2026年2月 核心变量（不改变你要的变量名）
db_name = "2026年2月盈亏表数据"
file_time_str = "2026-02-28"
file_date = datetime.strptime(file_time_str, '%Y-%m-%d')
month = "02"

# 连接数据库
try:
    conn = pymysql.connect(
        host='192.168.30.51',
        user='mysql',
        password='QL132.465',
        database=db_name,
        charset='utf8mb4'
    )
    print("✅ 成功连接数据库：", db_name)

except pymysql.MySQLError as e:
    print("❌ 数据库连接失败：", e)

✅ 成功连接数据库： 2026年2月盈亏表数据


In [99]:
zt1 = pd.read_sql('select 聚水潭店铺编号, 主体, 变更日期, 新主体 from 店铺主体', conn)
zt2 = pd.read_sql('select * from 主体身份', conn)
zt1['变更日期'] = pd.to_datetime(zt1['变更日期'], errors='coerce').dt.strftime('%Y-%m-%d')
yhz_1 = pd.read_excel(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\期末数据\聚合和管家\货款账户\2026.{12 if int(month) <1 else int(month)}月-店铺期末余额.xlsx').rename(columns = {'Unnamed: 0':'店铺ID', f'{12 if int(month) <=1 else int(month)}月货款账户期初':'期初余额', f'{int(month)}月货款账户期末':'期末余额'})
yhz_2 = pd.read_excel(r'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\期末数据\支付宝余额2025.12-2026.4.xlsx')

## 合伙人汇总

### 拼多多

In [129]:
ff_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\拼多多\拼多多_货款明细.csv', low_memory=False)

In [130]:
ff_2_grouped = ff_2.copy()

In [131]:
ff_2_grouped['金额1'] = np.where(
    ~(
        (ff_2_grouped['类型'].str.contains('现金抵减推广|销售额|申述补回', na = False)) & (ff_2_grouped['来源文件'].isin(['其他', '拼多多_货款明细']))
        |
        (ff_2_grouped['类型'].str.contains('直通车-收入', na = False)) & (ff_2_grouped['来源文件'] == '拼多多_推广账户')
    )
    & (ff_2_grouped['金额'] != 0)
    , -ff_2_grouped['金额'], ff_2_grouped['金额'])

In [132]:
ff_2_1 = ff_2_grouped.groupby(['店铺ID', '来源文件'])['金额1'].sum().reset_index().rename(columns = {'金额1':'本月累计'})

In [133]:
ff_2_3 = pd.merge(ff_2_grouped, ff_2_1, on = ['店铺ID', '来源文件'], how = 'left')

In [134]:
ff_2_3['本月累计'] = np.where(ff_2_3['类型'] == '扣款_现金抵减推广', 0, ff_2_3['本月累计'])

In [135]:
ff_2_3 = ff_2_3.fillna(0)

#### 货款明细和保证金期初期末

In [136]:
yhz_hk = yhz_1.groupby(['店铺ID'])[['期初余额', '期末余额']].sum().reset_index().rename(columns = {'期初余额':'货款账户期初', '期末余额':'货款账户期末'})
yhz_hk['来源文件'] = '拼多多_货款明细'

In [137]:
yhz_1['保证金期初余额'] = yhz_1[f'{12 if int(month) <=1 else int(month)}月店铺保证金期初'] + yhz_1[f'{12 if int(month) <=1 else int(month)}月活动保证金期初']
yhz_1['保证金期末余额'] = yhz_1[f'{int(month)}月店铺保证金期末'] + yhz_1[f'{int(month)}月活动保证金期末']
yhz_bzj = yhz_1.groupby(['店铺ID'])[['保证金期初余额', '保证金期末余额']].sum().reset_index()
yhz_bzj['来源文件'] = '拼多多_保证金'

In [138]:
yhz_ye = pd.merge(yhz_hk, yhz_bzj, on = ['店铺ID', '来源文件'], how = 'outer')

In [139]:
yhz_ye = yhz_ye.fillna(0)

In [140]:
ff_3 = pd.merge(ff_2_3, yhz_ye, on = ['店铺ID', '来源文件'], how = 'left')
ff_3 = ff_3.fillna(0)
ff_3['期初余额'] = ff_3['货款账户期初'] + ff_3['保证金期初余额']
ff_3['期末余额'] = ff_3['货款账户期末'] + ff_3['保证金期末余额']

#### 推广期初期末

In [141]:
yhz_1['推广期初余额'] = yhz_1[f'{12 if int(month) <=1 else int(month)}月推广账户期初余额']
yhz_1['推广期末余额'] = yhz_1[f'{int(month)}月推广账户期末余额']
yhz_tg = yhz_1.groupby(['店铺ID'])[[f'推广期初余额', f'推广期末余额']].sum().reset_index()
yhz_tg['来源文件'] = '拼多多_推广账户'

In [142]:
ff_3_tg = pd.merge(ff_3, yhz_tg, on = ['店铺ID', '来源文件'], how = 'left')
ff_3_tg = ff_3_tg.fillna(0)

In [143]:
ff_3_tg['期初余额1'] = ff_3_tg['推广期初余额'] + ff_3_tg['期初余额']
ff_3_tg['期末余额1'] = ff_3_tg['推广期末余额'] + ff_3_tg['期末余额']

In [144]:
ff_3_tg['核对'] = (
    round((ff_3_tg['期初余额1'] + ff_3_tg['本月累计']) - ff_3_tg['期末余额1'], 2)
    # 修复：小于0.005的负数都变成0
    .apply(lambda x: 0.00 if -0.005 < x < 0 else x)
    .apply(lambda x: f"{x:,.2f}")
)

In [145]:
ff_3_tg['核对'] = ff_3_tg['核对'].astype(str).str.replace(',', '').astype(float).abs()

#### 计算现金净流量

In [146]:
ff_3_mx_sc = ff_3_tg[
    (~(ff_3_tg['类型'].isin(['提现', '扣款_现金抵减推广'])) & (ff_3_tg['来源文件'].isin(['其他', '拼多多_货款明细'])))
    |
    (ff_3_tg['来源文件'].isin(['拼多多_推广账户']))
].groupby(['店铺ID', '来源文件'])['金额1'].sum().reset_index().rename(columns = {'金额1':'现金净流量'})

In [147]:
ff_3_mx_sc

,店铺ID,来源文件,现金净流量
0,11931578,拼多多_推广账户,165.25
1,11931578,拼多多_货款明细,14001.99
2,11931589,拼多多_推广账户,-82.79
3,11931589,拼多多_货款明细,10810.38
4,11931620,拼多多_推广账户,-12.35
...,...,...,...
273,17879625,拼多多_货款明细,56802.26
274,18011557,拼多多_推广账户,-17.63
275,18011557,拼多多_货款明细,649.02
276,18230774,拼多多_推广账户,-365.39


#### 汇总

In [148]:
ff_3_1 = pd.merge(ff_3_tg, ff_3_mx_sc, on = ['店铺ID', '来源文件'], how = 'left')
ff_3_1 = ff_3_1.fillna(0)

In [149]:
# 按店铺分组，标记不是“每组第一行”的位置
mask = ff_3_1.groupby(['店铺ID', '来源文件'])['现金净流量'].cumcount() > 0

# 这些行清空（设为 NaN）
ff_3_1.loc[mask, '现金净流量'] = ''

In [150]:
ff_4 = ff_3_1.groupby(['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '来源文件'])[['金额', '期初余额1','本月累计', '期末余额1', '核对', '现金净流量']].sum().reset_index().rename(columns = {'期初余额1':'期初余额', '期末余额1':'期末余额'})

In [151]:
ff_4['资金方向'] = np.where(ff_4['类型'].str.contains('销售额|直通车-收入', na = False), '流入金额',
                            np.where(ff_4['类型'].str.contains('现金抵减推广'), '其他', '流出金额'))

In [152]:
ff_4[(ff_4['核对'] != 0)]

,平台,合伙人,品牌,店铺ID,身份,店铺名称,新主体,类型,来源文件,金额,期初余额,本月累计,期末余额,核对,现金净流量,资金方向


In [153]:
# 按店铺分组，标记不是“每组第一行”的位置
mask = ff_4.groupby(['店铺ID', '来源文件'])[['期初余额', '本月累计', '期末余额', '核对', '现金净流量']].cumcount() > 0

# 这些行清空（设为 NaN）
ff_4.loc[mask, ['期初余额', '本月累计', '期末余额', '核对', '现金净流量']] = ''

In [154]:
ff_4 = ff_4[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '现金净流量', '资金方向', '来源文件']]

In [155]:
ff_4[ff_4['来源文件'] == '拼多多_货款明细']

,平台,合伙人,品牌,店铺ID,身份,店铺名称,新主体,类型,金额,期初余额,本月累计,期末余额,核对,现金净流量,资金方向,来源文件
0,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,扣款-售后补偿消费者,23.00,14846.63,20848.59,35695.22,0.0,20848.59,流出金额,拼多多_货款明细
1,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,扣款-小额打款,109.77,,,,,,流出金额,拼多多_货款明细
2,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,扣款-延迟发货,435.00,,,,,,流出金额,拼多多_货款明细
3,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,扣款-虚假发货,90.00,,,,,,流出金额,拼多多_货款明细
4,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,扣款-运费补偿,36.54,,,,,,流出金额,拼多多_货款明细
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市锦汐云服饰有限公司,直通车-充值,8000.00,,,,,,流出金额,拼多多_货款明细
2147,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市锦汐云服饰有限公司,退款-优惠券退款,811.73,,,,,,流出金额,拼多多_货款明细
2148,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市锦汐云服饰有限公司,退款-订单退款,3753.58,,,,,,流出金额,拼多多_货款明细
2149,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市锦汐云服饰有限公司,销售额- 订单收入,19402.52,,,,,,流入金额,拼多多_货款明细


In [156]:
ff_4[ff_4['来源文件'] == '拼多多_推广账户']

,平台,合伙人,品牌,店铺ID,身份,店铺名称,新主体,类型,金额,期初余额,本月累计,期末余额,核对,现金净流量,资金方向,来源文件
8,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,直通车-支出,2314.04,712.22,-314.04,398.18,0.0,-314.04,流出金额,拼多多_推广账户
9,拼多多,张优汇,白珍珠,14450954,小规模纳税人,白珍珠轻柔专卖店（白汇）,深圳市轻柔服饰有限公司,直通车-收入,2000.00,,,,,,流入金额,拼多多_推广账户
21,拼多多,张优汇,白珍珠,16645649,小规模纳税人,白珍珠优选棉专卖店（白汇）,深圳市优选棉服饰有限公司,直通车-支出,1493.97,129.42,106.03,235.45,0.0,106.03,流出金额,拼多多_推广账户
22,拼多多,张优汇,白珍珠,16645649,小规模纳税人,白珍珠优选棉专卖店（白汇）,深圳市优选棉服饰有限公司,直通车-收入,1600.00,,,,,,流入金额,拼多多_推广账户
38,拼多多,张优汇,红蜻蜓,16016133,小规模纳税人,红蜻蜓RED DRAGONFLY服饰配件旗舰店（红汇),深圳市优选棉服饰有限公司,直通车-支出,24076.73,767.84,-76.73,691.11,0.0,-76.73,流出金额,拼多多_推广账户
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2117,拼多多,陈镇鸿,红蜻蜓,16590437,小规模纳税人,红蜻蜓针织服饰旗舰店（红鸿）,深圳市锦汐云服饰有限公司,直通车-收入,2000.00,,,,,,流入金额,拼多多_推广账户
2130,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市诺初贸易有限公司普宁分公司,直通车-支出,2864.16,679.01,382.85,1061.86,0.0,382.85,流出金额,拼多多_推广账户
2131,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市诺初贸易有限公司普宁分公司,直通车-收入,3000.00,,,,,,流入金额,拼多多_推广账户
2145,拼多多,陈镇鸿,红蜻蜓,17792992,小规模纳税人,红蜻蜓服饰专卖店（红鸿）,深圳市锦汐云服饰有限公司,直通车-支出,7752.99,,,,,,流出金额,拼多多_推广账户


In [157]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/拼多多/拼多多合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    ff_4.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

汇总数据已保存到Excel文件: ../../结果/2026年2月/拼多多/拼多多合伙人汇总.xlsx

所有数据保存完成！


### 抖音

In [78]:
fy_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\抖音\抖音2.0(待处理).csv', low_memory=False)
yhz_gj = pd.read_excel(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\期末数据\2026.{int(month)}月-管家账户.xlsx', header = 2)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\群狼\\PycharmProjects\\PythonProject\\切换口径\\期末数据\\抖音\\2026.2月-抖音推广.xlsx'

In [ ]:
fy_2_grouped = fy_2.copy()

#### 推广期初期末

In [ ]:
yhz_dy_2 = yhz_1.groupby(['店铺ID'])[['推广期初余额', '推广期末余额']].sum().reset_index().rename(columns = {'推广期初余额':'期初余额', '推广期末余额':'期末余额'})
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
yhz_dy_2 = yhz_dy_2.fillna(0).infer_objects()
yhz_dy_2['来源文件'] = '抖音_日结报表'

#### 资金流水和保证金期初期末

In [ ]:
yhz_dy_hk = yhz_1.groupby(['店铺ID'])[['期初余额', '期末余额']].sum().reset_index().rename(columns = {'期初余额':'货款账户期初', '期末余额':'货款账户期末'})
yhz_dy_hk['来源文件'] = '抖音_资金流水明细'

In [ ]:
yhz_1['保证金期初余额'] = yhz_1[f'{int(month)}月店铺保证金期初'] + yhz_1[f'{int(month)}月活动保证金期初']
yhz_1['保证金期末余额'] = yhz_1[f'{int(month)}月店铺保证金期末'] + yhz_1[f'{int(month)}月活动保证金期末']
yhz_dy_bzj = yhz_1.groupby(['店铺ID'])[['保证金期初余额', '保证金期末余额']].sum().reset_index()
yhz_dy_bzj['来源文件'] = '抖音_保证金'

In [ ]:
yhz_dy_ye = pd.merge(yhz_dy_hk, yhz_dy_bzj, on = ['店铺ID', '来源文件'], how = 'outer')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
yhz_dy_ye = yhz_dy_ye.fillna(0).infer_objects()

In [ ]:
fy_3 = pd.merge(fy_2_grouped, yhz_dy_ye, on = ['店铺ID', '来源文件'], how = 'left')
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fy_3 = fy_3.fillna(0).infer_objects()
fy_3['期初余额'] = fy_3['货款账户期初'] + fy_3['保证金期初余额']
fy_3['期末余额'] = fy_3['货款账户期末'] + fy_3['保证金期末余额']

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fy_3 = fy_3.fillna(0).infer_objects()

In [ ]:
fy_4_2 = pd.merge(fy_3, yhz_dy_2, on = ['店铺ID', '来源文件'], how = 'left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fy_4_2 = fy_4_2.fillna(0).infer_objects()

In [ ]:
fy_4_2['期初余额'] = round(fy_4_2['期初余额_x'] + fy_4_2['期初余额_y'], 2)
fy_4_2['期末余额'] = round(fy_4_2['期末余额_x'] + fy_4_2['期末余额_y'], 2)

#### 管家账户期初期末

In [ ]:
yhz_gj.rename(columns = {'核算维度编码':'店铺ID', '借方':'管家期初余额', '借方.3': '管家期末余额'}, inplace = True)

In [ ]:
yhz_dy_gj = yhz_gj.groupby(['店铺ID'])[['管家期初余额', '管家期末余额']].sum().reset_index()
yhz_dy_gj['来源文件'] = '抖音_管家账户'

In [ ]:
fy_4_2_gj = pd.merge(fy_4_2, yhz_dy_gj, on = ['店铺ID', '来源文件'], how = 'left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fy_4_2_gj = fy_4_2_gj.fillna(0).infer_objects()

In [ ]:
fy_4_2_gj['期初余额1'] = round(fy_4_2_gj['期初余额'] + fy_4_2_gj['管家期初余额'], 2)
fy_4_2_gj['期末余额1'] = round(fy_4_2_gj['期末余额'] + fy_4_2_gj['管家期末余额'], 2)

In [ ]:
fy_4_2_gj.columns

In [ ]:
fy_4_2_gj = fy_4_2_gj[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额', '来源文件', '期初余额1', '期末余额1']].rename(columns = {'期初余额1':'期初余额', '期末余额1':'期末余额'})

#### 汇总

In [ ]:
fy_4_2_gj_1 = fy_4_2_gj.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '期末余额']].sum().reset_index()

In [ ]:
fy_4_2_gj_1['金额1'] = np.where(
    (~(fy_4_2_gj_1['类型'].str.contains('销售额', na=False) & (fy_4_2_gj_1['来源文件'] == '抖音_资金流水明细') & (fy_4_2_gj_1['金额'] != 0)) |
    (~(fy_4_2_gj_1['类型'].str.contains('账户余额充值|不可退返佣充值|平台赠款', na=False)) & (fy_4_2_gj_1['来源文件'] == '抖音_收支明细')  & (fy_4_2_gj_1['金额'] != 0)) |
    (~(fy_4_2_gj_1['类型'].str.contains('巨量千川-总存入', na=False)) & (fy_4_2_gj_1['来源文件'] == '抖音_日结报表')) & (fy_4_2_gj_1['金额'] != 0))
    , -fy_4_2_gj_1['金额'], fy_4_2_gj_1['金额'])

In [ ]:
fy_4_2_1 = fy_4_2_gj_1[fy_4_2_gj_1['类型'] != '巨量千川-总存入']

In [ ]:
fy_4_2_2 = fy_4_2_1.groupby(['店铺ID', '来源文件'])[['金额1']].sum().reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fy_4_3 = pd.merge(fy_4_2_gj , fy_4_2_2, on = ['店铺ID', '来源文件'], how = 'outer')

In [ ]:
import pandas as pd

# 复制一份避免修改原数据（可选，更安全）
df = fy_4_3.copy()
# 存储异常行的列表
error_rows = []

# 按店铺ID逐个判断
for shop_id in df['店铺ID'].unique():
    # 筛选当前店铺数据
    shop_data = df[df['店铺ID'] == shop_id]

    # 计算三个金额
    amt1 = shop_data.loc[shop_data['类型'] == '巨量千川-非赠款消耗', '金额'].sum()
    amt2 = shop_data.loc[shop_data['类型'] == '巨量千川-赠款消耗', '金额'].sum()
    amt3 = shop_data.loc[shop_data['类型'] == '巨量千川-余额总消耗', '金额'].sum()

    # ✅ 正确判断：浮点数用绝对值差判断，避免精度错误
    # 相差小于 0.01 元（1分钱）就认为相等
    if abs(amt1 + amt2 - amt3) > 0.01:
        # 追加异常行（用列表收集，最后统一合并，这是 pandas 最佳实践）
        error_rows.append({
            '店铺ID': shop_id,
            '类型': '结算与资金异常',
            '金额': 1,
            '来源文件': '结算与资金异常'
        })

# ✅ 最后统一把异常行追加到原数据
if error_rows:
    error_df = pd.DataFrame(error_rows)
    fy_4_3 = pd.concat([fy_4_3, error_df], ignore_index=True)

In [ ]:
columns_to_fill = ["店铺名称", "新主体", "身份", '平台', '合伙人', "品牌"]

# 按店铺ID分组 → 同店铺内部向下填充（最正确）
fy_4_3[columns_to_fill] = fy_4_3.groupby('店铺ID')[columns_to_fill].fillna(method='ffill')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fy_4_3 = fy_4_3.fillna(0).infer_objects()

In [ ]:
# 1. 计算每个店铺的净金额
fy_4_3['本月累计_净'] = fy_4_3['店铺ID'].map(
    fy_4_3.groupby('店铺ID').apply(
        lambda x: x[x['类型'] == '巨量千川-总存入']['金额'].sum() - x[x['类型'] == '巨量千川-余额总消耗']['金额'].sum()
    )
)
# 2. 填本月累计
fy_4_3['本月累计'] = np.where(fy_4_3['来源文件'] == '抖音_日结报表', fy_4_3['本月累计_净'], fy_4_3['本月累计'])

# 3. 算核对
fy_4_3['核对'] = round(fy_4_3['期初余额'] + np.where(fy_4_3['来源文件'] != '抖音_日结报表', fy_4_3['本月累计'], fy_4_3['本月累计_净']) - fy_4_3['期末余额'],3)

fy_4_3['核对'] += 0.0
fy_4_3.drop(columns='本月累计_净', inplace=True)

In [ ]:
fy_4_3['金额'] = np.where(
    (fy_4_3['类型'].str.contains('退款', na=False))
    , abs(fy_4_3['金额']), fy_4_3['金额'])

In [ ]:
fy_4_3['资金方向'] = np.where(fy_4_3['类型'].str[:3] == '销售额', '流入金额', '流出金额')

In [ ]:
fy_4_4 = fy_4_3[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '资金方向', '期初余额','本月累计', '期末余额', '核对', '来源文件']].copy()

In [ ]:
fy_4_4.loc[fy_4_4['来源文件'] == '抖音_收支明细', ['本月累计','核对']] = 0

In [ ]:
fy_4_4[fy_4_4['核对'] != 0].店铺ID.value_counts()

In [ ]:
fy_4_4[(fy_4_4['核对'] != 0)]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/抖音/抖音合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fy_4_4.to_excel(writer, sheet_name='合伙人汇总(横)', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 淘系

In [ ]:
ft_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\淘系\淘系(待处理).csv', low_memory=False)
yhz_tx_1 = pd.read_excel(r'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\期末数据\2026.2月-税务账套.xlsx').rename(columns = {'Unnamed: 0':'店铺ID'})

In [ ]:
ft_2_grouped = ft_2.copy()

In [ ]:
ft_2_grouped['银行账号'] = ft_2_grouped['银行账号'].fillna('0')

In [ ]:
ft_3 = ft_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '银行账号', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
ft_3_1 = ft_3[['店铺名称', '店铺ID', '新主体', '身份', '银行账号', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
t = '其他货币资金-店铺过户'

for i, r in ft_3_1[ft_3_1['类型']==t].copy().iterrows():
    shop = r['店铺ID']
    curr = (r['新主体'], r['身份'])
    pairs = [tuple(x) for x in ft_3_1[ft_3_1['店铺ID']==shop][['新主体','身份']].drop_duplicates().values]

    if len(pairs)==2:
        other_e, other_i = [p for p in pairs if p!=curr][0]
        half = r['金额']/2

        ft_3_1.loc[i, '金额'] = half
        ft_3_1 = pd.concat([ft_3_1, pd.DataFrame([{'店铺ID':shop,'新主体':other_e,'身份':other_i,'类型':t,'金额':-half,'来源文件':'淘系_聚合账户'}])], ignore_index=True)

ft_3_gh = ft_3_1.copy()

In [ ]:
# 要填充的列
cols = ["店铺名称", '银行账号', "平台", "合伙人", "品牌"]

# 分组向下填充（0转空值再ffill）
ft_3_gh[cols] = ft_3_gh.replace(0, pd.NA).groupby(['店铺ID', '新主体', '身份', '来源文件'])[cols].fillna(method='ffill')

In [ ]:
ft_3_gh['金额1'] = np.where(
    # 规则1：不是 销售额|其他货币资金|其他收入 → 转负
    ~(ft_3_gh['类型'].str.contains('销售额|其他货币资金|其他收入', na=False) & (ft_3_gh['来源文件'].isin(['淘系_支付宝', '淘系_聚合账户'])))
    &
    # 规则2：不是 (淘系保证金+保证金充值) 组合 → 转负
    ~(ft_3_gh['类型'].str.contains('保证金-天猫保证金-充值|保证金-淘宝保证金-充值', na=False) & (ft_3_gh['来源文件'] == '淘系_保证金'))
    &
    # 规则3：不是 (淘系推广+万相台支付宝充值) 组合 → 转负
    ~(ft_3_gh['类型'].str.contains('万相台-支付宝充值-自动|万相台-支付宝充值-手动', na=False) & (ft_3_gh['来源文件'] == '淘系_推广账户'))
    , -ft_3_gh['金额'], ft_3_gh['金额']
)

In [ ]:
ft_3_2 = ft_3_gh.groupby(['店铺ID', '银行账号', '来源文件'])['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
ft_3_2[ft_3_2['店铺ID'] == 13938678]

In [ ]:
ft_3_3 = pd.merge(ft_3_gh, ft_3_2, on = ['店铺ID', '银行账号', '来源文件'], how = 'left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
ft_3_3 = ft_3_3.fillna(0).infer_objects()

#### 支付宝期初期末

In [ ]:
ft_4_4 = pd.merge(ft_3_3, yhz_2, left_on=['银行账号', '店铺ID'], right_on=['支付宝账号', '店铺ID'], how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
ft_4_4 = ft_4_4.fillna(0).infer_objects()
yhz_tx_1 = yhz_tx_1.fillna(0).infer_objects()

#### 聚合账户和保证金期初期末

In [ ]:
yhz_1_jh = yhz_tx_1.groupby(['店铺ID'])[[f'{int(month)}月货款账户期初', f'{int(month)}月货款账户期末']].sum().reset_index()
yhz_1_jh['来源文件'] = '淘系_聚合账户'

In [ ]:
yhz_tx_1['保证金期初余额'] = yhz_tx_1[f'{int(month)}月店铺保证金期初'] + yhz_tx_1[f'{int(month)}月活动保证金期初']
yhz_tx_1['保证金期末余额'] = yhz_tx_1[f'{int(month)}月店铺保证金期末'] + yhz_tx_1[f'{int(month)}月活动保证金期末']
yhz_1_1 = yhz_tx_1.groupby(['店铺ID'])[['保证金期初余额', '保证金期末余额']].sum().reset_index()
yhz_1_1['来源文件'] = '淘系_保证金'

In [ ]:
yhz_2_1 = pd.merge(yhz_1_jh, yhz_1_1, on = ['店铺ID', '来源文件'], how = 'outer')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
yhz_2_1 = yhz_2_1.fillna(0).infer_objects()

In [ ]:
ft_4_4_1 = pd.merge(ft_4_4, yhz_2_1, on = ['店铺ID', '来源文件'], how = 'left')
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
ft_4_4_1 = ft_4_4_1.fillna(0).infer_objects()
ft_4_4_1['期初余额'] = ft_4_4_1[f'{12 if int(month) <1 else int(month)}月期末'] + ft_4_4_1['保证金期初余额'] + ft_4_4_1[f'{int(month)}月货款账户期初']
ft_4_4_1['期末余额'] = ft_4_4_1[f'{int(month)}月期末'] + ft_4_4_1['保证金期末余额'] + ft_4_4_1[f'{int(month)}月货款账户期末']

In [ ]:
ft_4_4_1.loc[(ft_4_4_1['店铺ID'] == 12042857) & (ft_4_4_1['来源文件'] == '淘系_保证金'), '期初余额'] = 0

#### 推广期初期末

In [ ]:
yhz_1_tg = yhz_tx_1.groupby(['店铺ID'])[[f'{int(month)}月推广账户期初余额', f'{int(month)}月推广账户期末余额']].sum().reset_index()
yhz_1_tg['来源文件'] = '淘系_推广账户'

In [ ]:
ft_4_4_2 = pd.merge(ft_4_4_1, yhz_1_tg, on = ['店铺ID', '来源文件'], how = 'left')
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
ft_4_4_2 = ft_4_4_2.fillna(0).infer_objects()
ft_4_4_2['期初余额1'] = ft_4_4_2['期初余额'] + ft_4_4_2[f'{int(month)}月推广账户期初余额']
ft_4_4_2['期末余额1'] = ft_4_4_2['期末余额'] + ft_4_4_2[f'{int(month)}月推广账户期末余额']

In [ ]:
ft_4_4_2 = ft_4_4_2[['店铺名称', '店铺ID', '新主体', '身份', '银行账号', '平台', '合伙人', '品牌', '类型', '金额', '来源文件', '本月累计', '期初余额1', '期末余额1']].copy().rename(columns = {'期初余额1':'期初余额', '期末余额1':'期末余额'})

In [ ]:
ft_4_4_2

#### 汇总

In [ ]:
ft_4_4_2['核对'] = round((ft_4_4_2['期初余额'] + ft_4_4_2['本月累计']) - ft_4_4_2['期末余额'], 2)

In [ ]:
ft_4_5 = ft_4_4_2.groupby(['店铺名称', '店铺ID', '银行账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index().rename(columns={'银行账号':'支付宝账号'})

In [ ]:
ft_4_5 = ft_4_5[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '支付宝账号', '类型', '金额', '期初余额', '本月累计', '期末余额', '核对', '来源文件']]

In [79]:
ft_4_5 = ft_4_5[~((ft_4_5['类型'] == '未知') & (ft_4_5['金额'] == 0))]

NameError: name 'ft_4_5' is not defined

In [ ]:
ft_4_5[ft_4_5['核对'] != 0].店铺ID.value_counts()

In [ ]:
ft_4_5[(ft_4_5['核对'] != 0) & (ft_4_5['来源文件'] == '淘系_支付宝')]

In [ ]:
ft_4_5[(ft_4_5['核对'] != 0) & (ft_4_5['来源文件'] == '淘系_聚合账户')]

In [ ]:
ft_4_5[(ft_4_5['核对'] != 0) & (ft_4_5['来源文件'] == '淘系_保证金')]

In [ ]:
ft_4_5[(ft_4_5['核对'] != 0) & (ft_4_5['来源文件'] == '淘系_推广账户')]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/淘系/淘系合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    ft_4_5.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 淘工厂

In [ ]:
fg_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\淘工厂\淘工厂账单2.0(待处理).csv', low_memory=False)

In [ ]:
fg_2_grouped = fg_2.copy()

In [ ]:
# 按店铺ID分组，对结算账户列进行前向填充（组内向上填充）
fg_2_grouped['支付宝账号'] = fg_2_grouped.groupby('店铺ID')['支付宝账号'].bfill()

In [ ]:
fg_3 = fg_2_grouped.groupby(['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fg_3_1 = fg_3[['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fg_3_1['金额1'] = np.where(
    ~(fg_3_1['类型'].str.contains('销售额', na=False) & (fg_3_1['来源文件'] == '淘工厂_支付宝') & (fg_3_1['金额'] != 0))
    &
    ~(fg_3_1['类型'].str.contains('销售额', na=False) & (fg_3_1['来源文件'] == '淘工厂_微信') & (fg_3_1['金额'] != 0))
    &
    ~(fg_3_1['类型'].str.contains('万相台-自动充值', na=False) & (fg_3_1['来源文件'] == '淘工厂_推广账户') & (fg_3_1['金额'] != 0))
    , -fg_3_1['金额'], fg_3_1['金额'])

In [ ]:
fg_3_2 = fg_3_1.groupby(['店铺ID', '来源文件'])['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fg_3_2

In [ ]:
fg_3_3 = pd.merge(fg_3_1, fg_3_2, on = ['店铺ID', '来源文件'], how = 'left')

#### 资金账单和支付宝期初期末

In [ ]:
yhz_1[yhz_1['平台'] == '淘工厂']

In [ ]:
yhz_wx_1 = yhz_1.groupby(['店铺ID'])[['期初余额', '期末余额']].sum().reset_index()
yhz_wx_1['来源文件'] = '淘工厂_微信'

In [ ]:
fg_4_4 = pd.merge(fg_3_3, yhz_wx_1, on = ['店铺ID', '来源文件'], how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fg_4_4 = fg_4_4.fillna(0).infer_objects()

In [ ]:
yhz_zfb_2 = yhz_2.groupby(['店铺ID', '支付宝账号'])[[f'{12 if int(month) <1 else int(month)}月期末', f'{int(month)}月期末']].sum().reset_index()
yhz_zfb_2['来源文件'] = '淘工厂_支付宝'

In [ ]:
fg_4_5 = pd.merge(fg_4_4, yhz_zfb_2, left_on=['店铺ID', '支付宝账号', '来源文件'], right_on=['店铺ID', '支付宝账号', '来源文件'], how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fg_4_5 = fg_4_5.fillna(0).infer_objects()

In [ ]:
fg_4_5['期初余额1'] = fg_4_5['期初余额'] + fg_4_5[f'{12 if int(month) <1 else int(month)}月期末']
fg_4_5['期末余额1'] = fg_4_5['期末余额'] + fg_4_5[f'{int(month)}月期末']

In [ ]:
fg_4_5 = fg_4_5[['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',
       '来源文件', '金额1', '本月累计', '期初余额1', '期末余额1']].copy().rename(columns = {'期初余额1':'期初余额', '期末余额1':'期末余额'})

#### 推广期初期末

In [ ]:
yhz_tg_2 = yhz_1.groupby(['店铺ID'])[[f'{int(month)}月推广账户期初余额', f'{int(month)}月推广账户期末余额']].sum().reset_index()
yhz_tg_2['来源文件'] = '淘工厂_推广账户'

In [ ]:
fg_4_6 = pd.merge(fg_4_5, yhz_tg_2, on=['店铺ID', '来源文件'], how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fg_4_6 = fg_4_6.fillna(0).infer_objects()

In [ ]:
fg_4_6['期初余额1'] = fg_4_6['期初余额'] + fg_4_6[f'{int(month)}月推广账户期初余额']
fg_4_6['期末余额1'] = fg_4_6['期末余额'] + fg_4_6[f'{int(month)}月推广账户期末余额']

In [ ]:
fg_4_6['核对'] = round((fg_4_6['期初余额1'] + fg_4_6['本月累计']) - (fg_4_6['期末余额1']), 2)

In [ ]:
fg_4_7 = fg_4_6.groupby(['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额1', '本月累计', '期末余额1', '核对']].sum().reset_index().rename(columns={'期初余额1':'期初余额','期末余额1':'期末余额'})

In [ ]:
fg_4_7 = fg_4_7[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']].copy()

In [ ]:
fg_4_7[fg_4_7['核对'] != 0].店铺ID.value_counts()

In [ ]:
fg_4_7[(fg_4_7['核对'] != 0) & (fg_4_7['来源文件'] == '淘工厂_支付宝')]

In [ ]:
fg_4_7[(fg_4_7['核对'] != 0) & (fg_4_7['来源文件'] == '淘工厂_微信')]

In [ ]:
fg_4_7[(fg_4_7['核对'] != 0) & (fg_4_7['来源文件'] == '淘工厂_推广账户')]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/淘工厂/淘工厂合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fg_4_7.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 快手

In [ ]:
fk_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\快手\快手2.0(待处理).csv', low_memory=False)

In [ ]:
fk_2_grouped = fk_2.copy()

In [ ]:
fk_3 = fk_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fk_3_1 = fk_3[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fk_3_1['金额1'] = np.where(
    (~(fk_3_1['类型'].str.contains('销售额|资金账单-交易结算|资金账单-退款补结算', na=False)) & (fk_3_1['金额'] != 0)  & ~(fk_3_1['类型'].isin(['结算账单-货款结算', '资金账单-交易结算'])))
    , -fk_3_1['金额'], fk_3_1['金额'])

In [ ]:
fk_3_2 = fk_3_1.groupby(['店铺ID', '来源文件'])['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fk_3_3 = pd.merge(fk_3_1, fk_3_2, on = ['店铺ID', '来源文件'], how = 'left')

In [ ]:
fk_4_4 = pd.merge(fk_3_3, ye1, left_on='店铺ID', right_on='ID', how='left').rename(columns={'货款账户期末':'期初余额'}).drop(['ID'], axis=1)

In [ ]:
fk_4_5 = pd.merge(fk_4_4, ye2, left_on='店铺ID', right_on='ID', how='left').rename(columns={'货款账户期末':'期末余额'}).drop(['ID'], axis=1)

In [ ]:
# 按 店铺ID 逐个判断
for shop_id in fk_4_5['店铺ID'].unique():
    # 取出当前店铺的两个金额
    amt1 = fk_4_5.loc[(fk_4_5['店铺ID']==shop_id) & (fk_4_5['类型']=='结算账单-货款结算'), '金额'].sum()
    amt2 = fk_4_5.loc[(fk_4_5['店铺ID']==shop_id) & (fk_4_5['类型']=='资金账单-交易结算'), '金额'].sum()
    # 不相等 → 直接追加一行
    if amt1 != amt2:
        fk_4_5.loc[len(fk_4_5)] = pd.Series({
            '店铺ID': shop_id,
            '类型': '结算与资金异常',
            '金额': 1,
            '来源文件': '结算与资金异常'
        })

In [ ]:
columns_to_fill = ["店铺名称", "新主体", "身份", '平台', '合伙人', "品牌"]

# 按店铺ID分组 → 同店铺内部向下填充（最正确）
fk_4_5[columns_to_fill] = fk_4_5.groupby('店铺ID')[columns_to_fill].fillna(method='ffill')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fk_4_5 = fk_4_5.fillna(0).infer_objects()

In [ ]:
fk_4_5['核对'] = round((fk_4_5['期初余额'] + fk_4_5['本月累计']) - fk_4_5['期末余额'], 2)

In [ ]:
fk_4_6 = fk_4_5.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index()

In [ ]:
fk_4_6 = fk_4_6[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']]

In [ ]:
fk_4_6.loc[fk_4_6['来源文件'] == '快手_结算账单', ['期初余额','本月累计','期末余额', '核对']] = 0.0

In [ ]:
fk_4_6[(fk_4_6['核对'] != 0) | (fk_4_6['类型'] == '结算与资金异常')].店铺ID.value_counts()

In [ ]:
fk_4_6[(fk_4_6['核对'] != 0) | (fk_4_6['类型'] == '结算与资金异常')]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/快手/快手合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fk_4_6.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 小红书

In [ ]:
fh_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\小红书\小红书2.0(待处理).csv', low_memory=False)

In [ ]:
fh_2_grouped = fh_2.copy()

In [ ]:
fh_3 = fh_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fh_3_1 = fh_3[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fh_3_1['金额1'] = np.where(
    (~(fh_3_1['类型'].str.contains('销售额', na=False)) & (fh_3_1['金额'] != 0)  & ~(fh_3_1['类型'].isin(['结算账单-货款结算', '资金账单-交易结算'])))
    , -fh_3_1['金额'], fh_3_1['金额'])

In [ ]:
fh_3_1_1 = fh_3_1[~(fh_3_1['类型'].isin(['结算账单-货款结算', '资金账单-交易结算']))]

In [ ]:
fh_3_2 = fh_3_1_1.groupby('店铺ID')['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fh_3_3 = pd.merge(fh_3_1, fh_3_2, on = '店铺ID', how = 'left')

In [ ]:
fh_4 = pd.merge(fh_3_3, ye1, left_on='店铺ID', right_on='ID', how='left').rename(columns={'货款账户期末':'期初余额'})

In [ ]:
fh_4.drop(['ID'], axis=1, inplace=True)

In [ ]:
fh_4_1 = pd.merge(fh_4, ye2, left_on='店铺ID', right_on='ID', how='left').rename(columns={'货款账户期末':'期末余额'})

In [ ]:
fh_4_1.drop(['ID'], axis=1, inplace=True)

In [ ]:
fh_4_1['核对'] = round((fh_4_1['期初余额'] + fh_4_1['本月累计']) - fh_4_1['期末余额'], 2)

In [80]:
fh_4_2 = fh_4_1.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index()

NameError: name 'fh_4_1' is not defined

In [ ]:
fh_4_2 = fh_4_2[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']]

In [ ]:
fh_4_2[fh_4_2['核对'] != 0].店铺ID.value_counts()

In [ ]:
fh_4_2[fh_4_2['核对'] != 0]

In [ ]:
fh_4_2

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/小红书/小红书合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fh_4_2.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 得物

In [ ]:
fd_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\得物\得物2.0(待处理).csv', low_memory=False)

In [ ]:
fd_2_grouped = fd_2.copy()

In [ ]:
fd_3 = fd_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fd_3_1 = fd_3[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fd_3_1['金额1'] = np.where(
    (~(fd_3_1['类型'].str.contains('销售额', na=False))
     & (fd_3_1['金额'] != 0))
    & ~(fd_3_1['类型'].isin(['应结金额']))
    , -fd_3_1['金额'], fd_3_1['金额'])

In [ ]:
fd_3_1_1 = fd_3_1[fd_3_1['类型'] != '应结金额']

In [ ]:
fd_3_2 = fd_3_1_1.groupby('店铺ID')['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fd_3_2

In [ ]:
fd_3_3 = pd.merge(fd_3_1, fd_3_2, on = '店铺ID', how = 'left')

In [ ]:
fd_3_3.drop(['金额1'], axis=1, inplace=True)

In [ ]:
fd_4 = pd.merge(fd_3_3, yhz_1, left_on='店铺ID', right_on='店铺ID', how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fd_4 = fd_4.fillna(0).infer_objects()

In [ ]:
import pandas as pd
import numpy as np

# 条件1：仅 【类型=应结金额 且 金额 != 本月累计】 → 需要追加异常行
condition = (fd_4['类型'] == '应结金额') & (fd_4['金额'] != fd_4['本月累计'])

# 筛选出符合条件1的行（只有这些行会新增异常行）
abnormal_df = fd_4[condition].copy()

# 给这些行生成 结算与资金异常 新行
if not abnormal_df.empty:
    abnormal_df['类型'] = '结算与资金异常'
    abnormal_df['金额'] = 0
    abnormal_df['来源文件'] = '结算与资金异常'

# 合并：原数据不动 + 只追加满足条件1的异常行
fd_4_cs = pd.concat([fd_4, abnormal_df], ignore_index=True)

In [ ]:
fd_4_cs['核对'] = round((fd_4_cs['期初余额'] + fd_4_cs['本月累计']) - fd_4_cs['期末余额'], 2)

In [ ]:
fd_4_2 = fd_4_cs.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index()

In [ ]:
fd_4_2 = fd_4_2[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']]

In [ ]:
# fd_4_2.loc[fd_4_2['来源文件'] == '得物_账单总览', ['期初余额','本月累计','期末余额', '核对']] = 0.0

In [ ]:
fd_4_2[(fd_4_2['核对'] != 0) | (fd_4_2['类型'] == '结算与资金异常')].店铺ID.value_counts()

In [ ]:
fd_4_2[(fd_4_2['核对'] != 0) | (fd_4_2['类型'] == '结算与资金异常')]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/得物/得物合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fd_4_2.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 微信小店

In [ ]:
fw_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\微信小店\微信小店2.0(待处理).csv', low_memory=False)

In [ ]:
fw_2_grouped = fw_2.copy()

In [ ]:
fw_3 = fw_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fw_3_1 = fw_3[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fw_3_1['金额1'] = np.where(
    (~(fw_3_1['类型'].str.contains('销售额', na=False))
     & (fw_3_1['金额'] != 0))
    , -fw_3_1['金额'], fw_3_1['金额'])

In [ ]:
fw_3_2 = fw_3_1.groupby('店铺ID')['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fw_3_2

In [ ]:
fw_3_3 = pd.merge(fw_3_1, fw_3_2, on = '店铺ID', how = 'left')

In [ ]:
fw_3_3.drop(['金额1'], axis=1, inplace=True)

In [ ]:
fw_4 = pd.merge(fw_3_3, yhz_1, left_on='店铺ID', right_on='店铺ID', how='left')

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fw_4 = fw_4.fillna(0).infer_objects()

In [ ]:
fw_4['核对'] = round((fw_4['期初余额'] + fw_4['本月累计']) - fw_4['期末余额'], 2)

In [ ]:
fw_4_1 = fw_4.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index()

In [ ]:
fw_4_1 = fw_4_1[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']]

In [ ]:
fw_4_1[(fw_4_1['核对'] != 0)].店铺ID.value_counts()

In [ ]:
fw_4_1[(fw_4_1['核对'] != 0)]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/微信小店/微信小店合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fw_4_1.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")

### 阿里

In [ ]:
fa_2 = pd.read_csv(fr'C:\Users\群狼\PycharmProjects\PythonProject\切换口径\结果\2026年{int(month)}月\阿里\阿里2.0(待处理).csv', low_memory=False)

In [ ]:
fa_2_grouped = fa_2.copy()

In [ ]:
fa_3 = fa_2_grouped.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '来源文件'])[['金额']].sum().reset_index()

In [ ]:
fa_3_1 = fa_3[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌',  '类型', '金额', '来源文件']].copy()

In [ ]:
fa_3_1['金额1'] = np.where(
    (~(fa_3_1['类型'].str.contains('销售额', na=False))
     & (fa_3_1['金额'] != 0))
    , -fa_3_1['金额'], fa_3_1['金额'])

In [ ]:
fa_3_2 = fa_3_1.groupby('店铺ID')['金额1'].sum().round(2).reset_index().rename(columns={'金额1':'本月累计'})

In [ ]:
fa_3_3 = pd.merge(fa_3_1, fa_3_2, on = '店铺ID', how = 'left')

In [ ]:
fa_3_3.drop(['金额1'], axis=1, inplace=True)

In [ ]:
fa_4 = pd.merge(fa_3_3, yhz_2, left_on='店铺ID', right_on='店铺ID', how='left').rename(columns={f'{12 if int(month) <1 else int(month)}月期末':'期初余额', f'{int(month)}月期末':'期末余额'})

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fa_4 = fa_4.fillna(0).infer_objects()

In [ ]:
fa_4['核对'] = round((fa_4['期初余额'] + fa_4['本月累计']) - fa_4['期末余额'], 2)

In [ ]:
fa_4_1 = fa_4.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额', '期初余额', '本月累计', '期末余额', '核对']].sum().reset_index()

In [ ]:
fa_4_1 = fa_4_1[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '期初余额','本月累计', '期末余额', '核对', '来源文件']]

In [ ]:
fa_4_1[(fa_4_1['核对'] != 0)].店铺ID.value_counts()

In [ ]:
fa_4_1[(fa_4_1['核对'] != 0)]

In [ ]:
import os
import pandas as pd
# ---------------------- 处理汇总数据：仍保存为Excel ----------------------
# 定义Excel文件路径
excel_path = fr'../../结果/2026年{int(month)}月/阿里/阿里合伙人汇总.xlsx'

# 检查文件是否存在，如果存在则删除
if os.path.exists(excel_path):
    os.remove(excel_path)

# 创建ExcelWriter对象，用于写入多个工作表（仅汇总数据）
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fa_4_1.to_excel(writer, sheet_name='合伙人汇总', index=False)

print(f"汇总数据已保存到Excel文件: {excel_path}")
print("\n所有数据保存完成！")